In [91]:
import pandas as pd
import nltk
import csv
import sys
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import math

In [58]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\samya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\samya\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\samya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\samya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [59]:
csv.field_size_limit(sys.maxsize)

9223372036854775807

In [60]:
df = pd.read_csv('C:/Users/samya/Downloads/Pandas Data/Practicals DSBDA/train.csv')
df

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1
...,...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,0
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,0
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,0
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...",1


In [61]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20800 non-null  int64 
 1   title   20242 non-null  object
 2   author  18843 non-null  object
 3   text    20761 non-null  object
 4   label   20800 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 812.6+ KB


In [63]:
df.describe()

,id,label
count,20800.000000,20800.000000
mean,10399.500000,0.500625
std,6004.587135,0.500012
min,0.000000,0.000000
25%,5199.750000,0.000000
50%,10399.500000,1.000000
75%,15599.250000,1.000000
max,20799.000000,1.000000


In [64]:
df['text'] = df['text'].fillna("")

In [65]:
text = df['text'][0]
text

'House Dem Aide: We Didn’t Even See Comey’s Letter Until Jason Chaffetz Tweeted It By Darrell Lucus on October 30, 2016 Subscribe Jason Chaffetz on the stump in American Fork, Utah ( image courtesy Michael Jolley, available under a Creative Commons-BY license) \nWith apologies to Keith Olbermann, there is no doubt who the Worst Person in The World is this week–FBI Director James Comey. But according to a House Democratic aide, it looks like we also know who the second-worst person is as well. It turns out that when Comey sent his now-infamous letter announcing that the FBI was looking into emails that may be related to Hillary Clinton’s email server, the ranking Democrats on the relevant committees didn’t hear about it from Comey. They found out via a tweet from one of the Republican committee chairmen. \nAs we now know, Comey notified the Republican chairmen and Democratic ranking members of the House Intelligence, Judiciary, and Oversight committees that his agency was reviewing emai

In [66]:
text = str(text)
print(text[:200])

House Dem Aide: We Didn’t Even See Comey’s Letter Until Jason Chaffetz Tweeted It By Darrell Lucus on October 30, 2016 Subscribe Jason Chaffetz on the stump in American Fork, Utah ( image courtesy Mic


In [67]:
sentence = sent_tokenize(text)
sentence

['House Dem Aide: We Didn’t Even See Comey’s Letter Until Jason Chaffetz Tweeted It By Darrell Lucus on October 30, 2016 Subscribe Jason Chaffetz on the stump in American Fork, Utah ( image courtesy Michael Jolley, available under a Creative Commons-BY license) \nWith apologies to Keith Olbermann, there is no doubt who the Worst Person in The World is this week–FBI Director James Comey.',
 'But according to a House Democratic aide, it looks like we also know who the second-worst person is as well.',
 'It turns out that when Comey sent his now-infamous letter announcing that the FBI was looking into emails that may be related to Hillary Clinton’s email server, the ranking Democrats on the relevant committees didn’t hear about it from Comey.',
 'They found out via a tweet from one of the Republican committee chairmen.',
 'As we now know, Comey notified the Republican chairmen and Democratic ranking members of the House Intelligence, Judiciary, and Oversight committees that his agency was

In [68]:
sentence = sent_tokenize(text)
sentence

['House Dem Aide: We Didn’t Even See Comey’s Letter Until Jason Chaffetz Tweeted It By Darrell Lucus on October 30, 2016 Subscribe Jason Chaffetz on the stump in American Fork, Utah ( image courtesy Michael Jolley, available under a Creative Commons-BY license) \nWith apologies to Keith Olbermann, there is no doubt who the Worst Person in The World is this week–FBI Director James Comey.',
 'But according to a House Democratic aide, it looks like we also know who the second-worst person is as well.',
 'It turns out that when Comey sent his now-infamous letter announcing that the FBI was looking into emails that may be related to Hillary Clinton’s email server, the ranking Democrats on the relevant committees didn’t hear about it from Comey.',
 'They found out via a tweet from one of the Republican committee chairmen.',
 'As we now know, Comey notified the Republican chairmen and Democratic ranking members of the House Intelligence, Judiciary, and Oversight committees that his agency was

In [69]:
word = word_tokenize(text)
word

['House',
 'Dem',
 'Aide',
 ':',
 'We',
 'Didn',
 '’',
 't',
 'Even',
 'See',
 'Comey',
 '’',
 's',
 'Letter',
 'Until',
 'Jason',
 'Chaffetz',
 'Tweeted',
 'It',
 'By',
 'Darrell',
 'Lucus',
 'on',
 'October',
 '30',
 ',',
 '2016',
 'Subscribe',
 'Jason',
 'Chaffetz',
 'on',
 'the',
 'stump',
 'in',
 'American',
 'Fork',
 ',',
 'Utah',
 '(',
 'image',
 'courtesy',
 'Michael',
 'Jolley',
 ',',
 'available',
 'under',
 'a',
 'Creative',
 'Commons-BY',
 'license',
 ')',
 'With',
 'apologies',
 'to',
 'Keith',
 'Olbermann',
 ',',
 'there',
 'is',
 'no',
 'doubt',
 'who',
 'the',
 'Worst',
 'Person',
 'in',
 'The',
 'World',
 'is',
 'this',
 'week–FBI',
 'Director',
 'James',
 'Comey',
 '.',
 'But',
 'according',
 'to',
 'a',
 'House',
 'Democratic',
 'aide',
 ',',
 'it',
 'looks',
 'like',
 'we',
 'also',
 'know',
 'who',
 'the',
 'second-worst',
 'person',
 'is',
 'as',
 'well',
 '.',
 'It',
 'turns',
 'out',
 'that',
 'when',
 'Comey',
 'sent',
 'his',
 'now-infamous',
 'letter',
 'anno

In [70]:
stop_words = stopwords.words('english')
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [71]:
filter = []
for w in word:
    if w.lower() not in stop_words:
        filter.append(w)

filter

['House',
 'Dem',
 'Aide',
 ':',
 '’',
 'Even',
 'See',
 'Comey',
 '’',
 'Letter',
 'Jason',
 'Chaffetz',
 'Tweeted',
 'Darrell',
 'Lucus',
 'October',
 '30',
 ',',
 '2016',
 'Subscribe',
 'Jason',
 'Chaffetz',
 'stump',
 'American',
 'Fork',
 ',',
 'Utah',
 '(',
 'image',
 'courtesy',
 'Michael',
 'Jolley',
 ',',
 'available',
 'Creative',
 'Commons-BY',
 'license',
 ')',
 'apologies',
 'Keith',
 'Olbermann',
 ',',
 'doubt',
 'Worst',
 'Person',
 'World',
 'week–FBI',
 'Director',
 'James',
 'Comey',
 '.',
 'according',
 'House',
 'Democratic',
 'aide',
 ',',
 'looks',
 'like',
 'also',
 'know',
 'second-worst',
 'person',
 'well',
 '.',
 'turns',
 'Comey',
 'sent',
 'now-infamous',
 'letter',
 'announcing',
 'FBI',
 'looking',
 'emails',
 'may',
 'related',
 'Hillary',
 'Clinton',
 '’',
 'email',
 'server',
 ',',
 'ranking',
 'Democrats',
 'relevant',
 'committees',
 '’',
 'hear',
 'Comey',
 '.',
 'found',
 'via',
 'tweet',
 'one',
 'Republican',
 'committee',
 'chairmen',
 '.',
 'kn

In [72]:
clean = [c for c in filter if c.isalnum()]
clean

['House',
 'Dem',
 'Aide',
 'Even',
 'See',
 'Comey',
 'Letter',
 'Jason',
 'Chaffetz',
 'Tweeted',
 'Darrell',
 'Lucus',
 'October',
 '30',
 '2016',
 'Subscribe',
 'Jason',
 'Chaffetz',
 'stump',
 'American',
 'Fork',
 'Utah',
 'image',
 'courtesy',
 'Michael',
 'Jolley',
 'available',
 'Creative',
 'license',
 'apologies',
 'Keith',
 'Olbermann',
 'doubt',
 'Worst',
 'Person',
 'World',
 'Director',
 'James',
 'Comey',
 'according',
 'House',
 'Democratic',
 'aide',
 'looks',
 'like',
 'also',
 'know',
 'person',
 'well',
 'turns',
 'Comey',
 'sent',
 'letter',
 'announcing',
 'FBI',
 'looking',
 'emails',
 'may',
 'related',
 'Hillary',
 'Clinton',
 'email',
 'server',
 'ranking',
 'Democrats',
 'relevant',
 'committees',
 'hear',
 'Comey',
 'found',
 'via',
 'tweet',
 'one',
 'Republican',
 'committee',
 'chairmen',
 'know',
 'Comey',
 'notified',
 'Republican',
 'chairmen',
 'Democratic',
 'ranking',
 'members',
 'House',
 'Intelligence',
 'Judiciary',
 'Oversight',
 'committees',

In [73]:
pos_tags = pos_tag(clean)
pos_tags

[('House', 'NNP'),
 ('Dem', 'NNP'),
 ('Aide', 'NNP'),
 ('Even', 'NNP'),
 ('See', 'NNP'),
 ('Comey', 'NNP'),
 ('Letter', 'NNP'),
 ('Jason', 'NNP'),
 ('Chaffetz', 'NNP'),
 ('Tweeted', 'NNP'),
 ('Darrell', 'NNP'),
 ('Lucus', 'NNP'),
 ('October', 'NNP'),
 ('30', 'CD'),
 ('2016', 'CD'),
 ('Subscribe', 'NNP'),
 ('Jason', 'NNP'),
 ('Chaffetz', 'NNP'),
 ('stump', 'JJ'),
 ('American', 'NNP'),
 ('Fork', 'NNP'),
 ('Utah', 'NNP'),
 ('image', 'NN'),
 ('courtesy', 'NN'),
 ('Michael', 'NNP'),
 ('Jolley', 'NNP'),
 ('available', 'JJ'),
 ('Creative', 'NNP'),
 ('license', 'NN'),
 ('apologies', 'NNS'),
 ('Keith', 'NNP'),
 ('Olbermann', 'NNP'),
 ('doubt', 'NN'),
 ('Worst', 'NNP'),
 ('Person', 'NNP'),
 ('World', 'NNP'),
 ('Director', 'NNP'),
 ('James', 'NNP'),
 ('Comey', 'NNP'),
 ('according', 'VBG'),
 ('House', 'NNP'),
 ('Democratic', 'JJ'),
 ('aide', 'RB'),
 ('looks', 'VBZ'),
 ('like', 'IN'),
 ('also', 'RB'),
 ('know', 'VBP'),
 ('person', 'NN'),
 ('well', 'RB'),
 ('turns', 'VBZ'),
 ('Comey', 'NNP'),
 ('se

In [78]:
stemmed = []
stemmer = SnowballStemmer('english')
for word in clean:
    stemmed.append(stemmer.stem(word))

stemmed

['hous',
 'dem',
 'aid',
 'even',
 'see',
 'comey',
 'letter',
 'jason',
 'chaffetz',
 'tweet',
 'darrel',
 'lucus',
 'octob',
 '30',
 '2016',
 'subscrib',
 'jason',
 'chaffetz',
 'stump',
 'american',
 'fork',
 'utah',
 'imag',
 'courtesi',
 'michael',
 'jolley',
 'avail',
 'creativ',
 'licens',
 'apolog',
 'keith',
 'olbermann',
 'doubt',
 'worst',
 'person',
 'world',
 'director',
 'jame',
 'comey',
 'accord',
 'hous',
 'democrat',
 'aid',
 'look',
 'like',
 'also',
 'know',
 'person',
 'well',
 'turn',
 'comey',
 'sent',
 'letter',
 'announc',
 'fbi',
 'look',
 'email',
 'may',
 'relat',
 'hillari',
 'clinton',
 'email',
 'server',
 'rank',
 'democrat',
 'relev',
 'committe',
 'hear',
 'comey',
 'found',
 'via',
 'tweet',
 'one',
 'republican',
 'committe',
 'chairmen',
 'know',
 'comey',
 'notifi',
 'republican',
 'chairmen',
 'democrat',
 'rank',
 'member',
 'hous',
 'intellig',
 'judiciari',
 'oversight',
 'committe',
 'agenc',
 'review',
 'email',
 'recent',
 'discov',
 'order'

In [82]:
lm = WordNetLemmatizer()
lemmatized = []
for word in clean:
    lemmatized.append(lm.lemmatize(word))
    
lemmatized

['House',
 'Dem',
 'Aide',
 'Even',
 'See',
 'Comey',
 'Letter',
 'Jason',
 'Chaffetz',
 'Tweeted',
 'Darrell',
 'Lucus',
 'October',
 '30',
 '2016',
 'Subscribe',
 'Jason',
 'Chaffetz',
 'stump',
 'American',
 'Fork',
 'Utah',
 'image',
 'courtesy',
 'Michael',
 'Jolley',
 'available',
 'Creative',
 'license',
 'apology',
 'Keith',
 'Olbermann',
 'doubt',
 'Worst',
 'Person',
 'World',
 'Director',
 'James',
 'Comey',
 'according',
 'House',
 'Democratic',
 'aide',
 'look',
 'like',
 'also',
 'know',
 'person',
 'well',
 'turn',
 'Comey',
 'sent',
 'letter',
 'announcing',
 'FBI',
 'looking',
 'email',
 'may',
 'related',
 'Hillary',
 'Clinton',
 'email',
 'server',
 'ranking',
 'Democrats',
 'relevant',
 'committee',
 'hear',
 'Comey',
 'found',
 'via',
 'tweet',
 'one',
 'Republican',
 'committee',
 'chairman',
 'know',
 'Comey',
 'notified',
 'Republican',
 'chairman',
 'Democratic',
 'ranking',
 'member',
 'House',
 'Intelligence',
 'Judiciary',
 'Oversight',
 'committee',
 'agenc

In [86]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform([' '.join(lemmatized)])
tfidf_matrix.shape

(1, 267)

In [88]:
corpus = df['text']

vectorizer = TfidfVectorizer(max_features=5000)

In [92]:
tf = Counter(clean)

total_words = len(clean)

tf_normal = {word: count/total_words for word,count in tf.items()}

print(tf_normal)

{'House': 0.014184397163120567, 'Dem': 0.004728132387706856, 'Aide': 0.002364066193853428, 'Even': 0.002364066193853428, 'See': 0.002364066193853428, 'Comey': 0.02364066193853428, 'Letter': 0.002364066193853428, 'Jason': 0.01182033096926714, 'Chaffetz': 0.03309692671394799, 'Tweeted': 0.002364066193853428, 'Darrell': 0.01182033096926714, 'Lucus': 0.004728132387706856, 'October': 0.004728132387706856, '30': 0.002364066193853428, '2016': 0.004728132387706856, 'Subscribe': 0.002364066193853428, 'stump': 0.002364066193853428, 'American': 0.002364066193853428, 'Fork': 0.002364066193853428, 'Utah': 0.004728132387706856, 'image': 0.002364066193853428, 'courtesy': 0.004728132387706856, 'Michael': 0.002364066193853428, 'Jolley': 0.002364066193853428, 'available': 0.002364066193853428, 'Creative': 0.002364066193853428, 'license': 0.002364066193853428, 'apologies': 0.002364066193853428, 'Keith': 0.002364066193853428, 'Olbermann': 0.002364066193853428, 'doubt': 0.002364066193853428, 'Worst': 0.004

In [94]:
documents = corpus.tolist()
N = len(documents)

word_doc_count = {}

for doc in documents:
    words = set(doc.split())
    for word in words:
        word_doc_count[word] = word_doc_count.get(word,0)+1

idf = {word: math.log(N/count) for word,count in word_doc_count.items()}

print('IDF:\n', dict(list(idf.items())[:20]))

IDF:
 {'Follow': 2.21649561518188, 'Dem': 6.279146619559763, 'a': 0.0813456394539524, 'Aide:': 9.94270826568941, 'Worst': 6.3591893272333, 'one': 0.5989740105426956, 'license)': 8.333270353255308, 'and': 0.08296363610120012, 'scared': 4.600374013724599, 'but': 0.5056308068888866, 'Jason': 4.007814070069822, 'we': 0.7661315238660358, 'into': 0.752978847617351, 'I': 0.8273381313044534, 'is': 0.1544632286655582, 'until': 1.82718738414264, 'so': 0.8683021709540597, 'Comey’s': 4.259128498350727, 'percent': 1.9692083016647794, 'With': 2.0498827394382917}


In [98]:
tfidf_matrix = vectorizer.fit_transform(documents)
tfidf_matrix.shape

(20800, 5000)

In [100]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns = vectorizer.get_feature_names_out()
)

In [101]:
tfidf_df

,00,000,10,100,11,12,125,13,14,15,...,как,мы,на,не,но,по,сша,то,что,это
0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.000000,0.0,0.016310,0.000000,0.0,0.020081,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.029306,0.000000,0.099468,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20795,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20796,0.0,0.000000,0.000000,0.0,0.015128,0.000000,0.0,0.000000,0.018691,0.015805,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20797,0.0,0.017190,0.017855,0.0,0.041468,0.022973,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20798,0.0,0.000000,0.036846,0.0,0.042787,0.000000,0.0,0.052681,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
